# WAXAL-Dual-Core: End-to-End Training Pipeline

**Eliminating the Tokenization Tax for African Languages via Dual-Stream Tokenization**

**Target Hardware:** Google Colab T4 (training) → Dell Latitude 7400 / 8GB RAM (edge deployment)

**Pipeline:**
1. Setup & Dependencies
2. Download WAXAL Dataset
3. Baseline Token Fertility
4. Train Router Classifier
5. Train Unified BPE Vocabulary
6. Train LoRA Variant (A–E)
7. Benchmark Fertility Reduction
8. Export to GGUF

## 1. Setup & Dependencies

In [1]:
from huggingface_hub import notebook_login

notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


In [2]:
# Check GPU
!nvidia-smi

Wed Apr  8 14:53:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
# Skip torch and psutil (already in Colab)
# Focus on the research-specific extensions
!uv pip install -U transformers datasets accelerate peft bitsandbytes llama-cpp-python

Using Python 3.12.13 environment at: /usr
Resolved 78 packages in 1.53s                                        
Prepared 41 packages in 8m 45s                                           
Uninstalled 23 packages in 1.14s
Installed 41 packages in 425ms                              
 - aiohttp==3.13.4
 + aiohttp==3.13.5
 + bitsandbytes==0.49.2
 - charset-normalizer==3.4.6
 + charset-normalizer==3.4.7
 - click==8.3.1
 + click==8.3.2
 - cuda-bindings==12.9.4
 + cuda-bindings==13.2.0
 - cuda-pathfinder==1.5.0
 + cuda-pathfinder==1.5.2
 - cuda-toolkit==12.8.1
 + cuda-toolkit==13.0.2
 - datasets==4.0.0
 + datasets==4.8.4
 - dill==0.3.8
 + dill==0.4.1
 + diskcache==5.6.3
 - fsspec==2025.3.0
 + fsspec==2026.2.0
 - hf-xet==1.4.2
 + hf-xet==1.4.3
 - huggingface-hub==1.8.0
 + huggingface-hub==1.9.2
 + llama-cpp-python==0.3.20
 - multiprocess==0.70.16
 + multiprocess==0.70.19
 - numpy==2.0.2
 + numpy==2.4.4
 + nvidia-cublas==13.1.0.3
 + nvidia-cuda-cupti==13.0.85
 + nvidia-cuda-nvrtc==13.0.88
 + nvid

In [4]:
import os
from pathlib import Path

if not os.path.exists('somax'):
    !git clone 'https://github.com/attabeezy/somax.git'
    %cd somax
else:
    %cd somax

Cloning into 'somax'...
remote: Enumerating objects: 84, done.
remote: Counting objects: 100% (84/84), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 84 (delta 38), reused 69 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (84/84), 56.51 KiB | 11.30 MiB/s, done.
Resolving deltas: 100% (38/38), done.
/content/somax


In [5]:
import os

# ── Configuration ─────────────────────────────────────────
LANGUAGE    = "akan"                      # proof-of-concept language
TRAIN_GROUP = "D"                         # primary hypothesis: TTS → ASR → TTS
BASE_MODEL  = "meta-llama/Llama-3.2-1B"
DATA_DIR    = f"data/{LANGUAGE}"
CONFIG_FILE = "configs/variants.yaml"
# ──────────────────────────────────────────────────────────

HF_TOKEN = os.environ.get("HF_TOKEN", "SIKE TOKEN NOT SET")
if not HF_TOKEN:
    print("WARNING: HF_TOKEN not set. Required for Llama models.")
    print("Set it with:  os.environ['HF_TOKEN'] = 'your_token'")

In [6]:
import shutil

def setup_drive() -> str | None:
    """Mount Google Drive and return results base path."""
    try:
        from google.colab import drive
        drive.mount("/content/drive", timeout_ms=120000)
        base = "/content/drive/MyDrive/WAXAL-results"
        os.makedirs(base, exist_ok=True)
        print(f"Google Drive mounted: {base}")
        return base
    except Exception as e:
        print(f"Google Drive not available: {e}")
        return None

def save_to_drive(source_dir: str, drive_base: str | None, label: str) -> None:
    """Copy a local directory to Google Drive."""
    if not drive_base or not os.path.exists(source_dir):
        return
    dest = os.path.join(drive_base, label)
    if os.path.exists(dest):
        shutil.rmtree(dest)
    shutil.copytree(source_dir, dest)
    print(f"Saved to Drive: {dest}")

DRIVE_BASE = setup_drive()

Mounted at /content/drive
Google Drive mounted: /content/drive/MyDrive/WAXAL-results


## 2. Download WAXAL Dataset

In [ ]:
!python scripts/download.py --lang {LANGUAGE} --output data/  # Uncomment if you downloaded data in this session and want to save it to Drive

Target directory: data/akan

ASR config: aka_asr
`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'google/WaxalNLP' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
README.md: 29.2kB [00:00, 17.8MB/s]
Resolving data files: 100% 72/72 [00:00<00:00, 31592.21it/s]
Resolving data files: 100% 270/270 [00:00<00:00, 19823.24it/s]
data/ASR/aka/aka-train-00000.parquet: 100% 629M/629M [00:47<00:00, 13.2MB/s] 
data/ASR/aka/aka-train-00001.parquet: 100% 628M/628M [00:45<00:00, 14.0MB/s] 
data/ASR/aka/aka-train-00002.parquet: 100% 632M/632M [00:42<00:00, 14.8MB/s] 
data/ASR/aka/aka-train-00003.parquet: 100% 626M/626M [00:58<00:00, 10.6MB/s] 
data/ASR/aka/aka-train-00004.parquet: 100% 625M/625M [00:52<00:00, 11.9MB/s] 
data/ASR/aka/aka-train-00005.parquet: 100% 33.4M/33.4M [00:16<00:00, 1.99MB/s]
data/ASR/aka/

In [ ]:
save_to_drive(DATA_DIR, DRIVE_BASE, "data") # Uncomment if you downloaded data in this session and want to save it to Drive

Saved to Drive: /content/drive/MyDrive/WAXAL-results/data


## 3. Baseline Token Fertility

In [ ]:
!python benchmark_fertility.py \
    --tokenizer {BASE_MODEL} \
    --test-file {DATA_DIR}/twi_tts_test.jsonl \
    --output results/baseline_fertility.json

Traceback (most recent call last):
  File "/content/somax/benchmark_fertility.py", line 196, in <module>
    main()
  File "/content/somax/benchmark_fertility.py", line 151, in main
    raise FileNotFoundError(f"Test file not found: {test_file}")
FileNotFoundError: Test file not found: data/akan/twi_tts_test.jsonl


: 

: 

## 4. Train Router Classifier

In [12]:
!python scripts/train_router.py \
    --data {DATA_DIR} \
    --output models/router/ \
    --language {LANGUAGE}

Training router classifier for language: akan
Data: data/akan
Loaded 872 TTS samples

Total samples: 872 (0 ASR, 872 TTS)
Training classifier...
Traceback (most recent call last):
  File "/content/somax/scripts/train_router.py", line 141, in <module>
    main()
  File "/content/somax/scripts/train_router.py", line 123, in main
    classifier = train_classifier(texts, labels)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/somax/scripts/train_router.py", line 91, in train_classifier
    scores = cross_val_score(pipeline, texts, labels, cv=5, scoring="f1_macro")
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/utils/_param_validation.py", line 216, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 684, in cross_val_score
    cv_results = cross_validate(
            

## 5. Train Unified BPE Vocabulary

In [13]:
!python scripts/train_bpe.py \
    --input {DATA_DIR} \
    --output models/tokenizers/ \
    --language {LANGUAGE} \
    --vocab-size 8000

TTS: 872 samples from data/akan/twi_tts_train.jsonl
Training unified BPE on 872 samples (0 ASR + 872 TTS)...
[00:00:00] Tokenize words                 ██████████████████ 7483     /     7483[00:00:00] Tokenize words                 ██████████████████ 0        /        0
[00:00:00] Count pairs                    ██████████████████ 7483     /     7483
[00:00:00] Compute merges                 ██████████████████ 7895     /     7895
Unified tokenizer saved to: models/tokenizers/akan/unified_tokenizer.json
Tokenizer config saved to: models/tokenizers/akan/tokenizer_config.json
Computing per-stream token statistics...
Stream token stats saved to: models/tokenizers/akan/stream_token_stats.json

Training stats saved to: models/tokenizers/akan/training_stats.json
Vocab size: 8000
Token dominance: {'tts': 7013, 'shared': 987}
Done.


## 6. Train LoRA Variant

Edit `TRAIN_GROUP` in the config cell above to run a different variant:

| Group | Sequence | Notes |
|-------|----------|-------|
| A | ASR only | |
| B | TTS only | |
| C | Mixed | |
| **D** | TTS → ASR → TTS | Primary hypothesis |
| E | ASR → TTS | |

In [ ]:
!python scripts/train_lora.py \
    --group {TRAIN_GROUP} \
    --model {BASE_MODEL} \
    --data {DATA_DIR} \
    --output checkpoints/ \
    --config {CONFIG_FILE}

In [ ]:
save_to_drive("checkpoints", DRIVE_BASE, "checkpoints")

## 7. Benchmark Fertility Reduction

In [ ]:
!python benchmark_fertility.py \
    --tokenizer {BASE_MODEL} \
    --waxal-tokenizer models/tokenizers/{LANGUAGE}/unified_tokenizer.json \
    --test-file {DATA_DIR}/twi_tts_test.jsonl \
    --compare \
    --output results/fertility_comparison.json

## 8. Export to GGUF (for Edge Deployment)

In [ ]:
!python scripts/export_gguf.py \
    --checkpoint checkpoints/variant_{TRAIN_GROUP}/final/ \
    --output models/gguf/ \
    --quantization Q4_K_M

In [ ]:
save_to_drive("results", DRIVE_BASE, "results")
save_to_drive("models", DRIVE_BASE, "models")

## Summary

In [ ]:
import json
from pathlib import Path

print("=" * 50)
print("WAXAL-Dual-Core Pipeline Complete!")
print("=" * 50)
print(f"Language:       {LANGUAGE}")
print(f"Variant:        {TRAIN_GROUP}")

results_path = Path("results/fertility_comparison.json")
if results_path.exists():
    r = json.loads(results_path.read_text())
    print(f"\nBaseline F:     {r['baseline']['fertility_mean']:.3f} tokens/word")
    print(f"WAXAL F:        {r['waxal']['fertility_mean']:.3f} tokens/word")
    print(f"Reduction:      {r['reduction_pct']:.1f}%  (target: ≥30%)")
else:
    baseline_path = Path("results/baseline_fertility.json")
    if baseline_path.exists():
        r = json.loads(baseline_path.read_text())
        print(f"\nBaseline F:     {r['fertility_mean']:.3f} tokens/word")

print("\nNext steps:")
print("  1. Download results from Google Drive")
print("  2. Run benchmark_inference.py on Dell Latitude 7400")
print("  3. Run benchmark_fertility.py --compare on edge hardware")

## Summary